In [3]:
import os
import json

from dotenv import load_dotenv

from azure.ai.ml import command, Input, Output
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

from utils import connect_to_workspace, get_blob_client, upload_to_blob, get_datastore_uri

load_dotenv()

# Inputs
CONFIG_FILE_PATH = "../infra/infra_config.json"
LOCAL_DATA_PATH = "C:\\Users\\m_kal\\Downloads\\datasets\\rossmann-store-sales"  # Local folder to upload (ensure it exists)
DESTINATION_BLOB_PATH = "rossmann/"  # Destination path inside the container


with open(CONFIG_FILE_PATH, "r") as f:
    config = json.load(f)

### Make the ML client

In [4]:
ml_client = connect_to_workspace(subscription_id=os.getenv("SUBSCRIPTION_ID"),
                                 resource_group=config["resource_group"],
                                 workspace_name=config["workspace_name"])

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


Successfully connected to Azure ML Workspace: mlw-workspace


### Upload local directory to Azure Blob Storage and register as a Data asset in Azure ML 

In [5]:
blob_client = get_blob_client(storage_account_name=config["storage_account_name"],
                              chunk_size=4 * 1024 * 1024)   # 4MB chunk size - If file is larger than this size we break it up in 
                                                            # chunks of this size and upload each chunk separately. 
                                                            # This is to avoid network timeouts on large files.

container_client = blob_client.get_container_client(config["container_name"])
failed_uploads = upload_to_blob(client=container_client,
                                source=LOCAL_DATA_PATH,
                                destination=DESTINATION_BLOB_PATH,
                                overwrite=True,     # Whether to overwrite existing files in the destination blob path.
                                max_concurrency=4,  # Number of parallel uploads to perform.
                                timeout=300)        # Timeout in seconds for each upload operation

Uploading sample_submission.csv...
Uploading store.csv...
Uploading test.csv...
Uploading train.csv...
All files uploaded successfully!


### Register the uploaded cloud path as a Data asset

In [6]:
my_data_asset = Data(name="artifacts",
                     version="2.0.0",
                     description="Dataset uploaded to the custom data_container_datastore",
                     path=get_datastore_uri(config["datastore_name"], DESTINATION_BLOB_PATH),
                     type=AssetTypes.URI_FOLDER)

registered_data_asset = ml_client.data.create_or_update(my_data_asset)
print(f"Data asset registered as: {registered_data_asset.name}")

Data asset registered as: artifacts


### Submit a dummy job

To verify that the data asset is accessible, we can create a simple command job that lists the contents of the data directory.
Upon successful execution, the job should store the list of files in the output data directory, confirming that the data asset is accessible and mounted correctly.
To see the job's output, you can check the Azure ML Studio or use the Azure ML SDK to retrieve the job's logs after it completes:
Go to the Azure ML Studio, navigate to the "Jobs" section, and find the job you submitted (search by display name below). 
Click on it to view its details and logs.

In [8]:
job = command(
    command="ls -1 ${{inputs.input_dir}} > ${{outputs.output_dir}}/filenames.txt",
    inputs={
        "input_dir": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{registered_data_asset.name}:{registered_data_asset.version}", 
            mode="ro_mount"
        ),
    },
    outputs={
        "output_dir": Output(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml://datastores/{config['datastore_name']}/paths/job_outputs/",
            mode="rw_mount"
        ),
    },
    compute=config["compute_name"],
    environment="azureml://registries/azureml/environments/sklearn-1.1/versions/4",
    display_name="datastore-access-test-2"
)

# 4. Submit the job
returned_job = ml_client.jobs.create_or_update(job)
print(f"Job submitted. Check URL: {returned_job.studio_url}")

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Job submitted. Check URL: https://ml.azure.com/runs/amusing_panda_cdxd7qsskr?wsid=/subscriptions/dc6fd0ed-8e9d-4a63-b4a6-cc23be43b154/resourcegroups/rg-ml-workspace-rossmann/workspaces/mlw-workspace&tid=bd335ce6-2a74-47f0-95f4-489d94f9cd99


# Submit job

In [ ]:

# 4. Define the Command Job configuration
job = command(
    display_name="Generic ML Pipeline Run",
    experiment_name="generic-jobs",
    description="A generic template for running code on Azure Spot ML Clusters",
    
    # Point to local directories and define execution script
    code="./src",  # Packs everything inside ./src (including main.py and config.yaml)

    command="python main.py --config debug_config.yaml --data_dir ${{inputs.input_dir}}",
    
    compute=config["compute_name"],
    
    # The software environment to run inside the container
    # List of curated environments: https://ml.azure.com/registries/azureml/environments
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    
    # Reference the freshly registered cloud dataset directly!
    inputs={
        "input_dir": Input(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml:{registered_data_asset.name}:{registered_data_asset.version}", 
            mode="ro_mount"
        ),
    },
    outputs={
        "output_dir": Output(
            type=AssetTypes.URI_FOLDER,
            path=f"azureml://datastores/{config['datastore_name']}/paths/job_outputs/",
            mode="rw_mount"
        ),
    },
    
    # Guardrail timeout (e.g., 3600 seconds = 1 hour limit)
    limits={"timeout": 3600} 
)


In [ ]:
# 5. Submit to Azure ML
print("Submitting command job...")
returned_job = ml_client.jobs.create_or_update(job)

print("Job Submitted successfully!")
print(f"Job Name:   {returned_job.name}")
print(f"Status:     {returned_job.status}")
print(f"Studio Link: {returned_job.studio_url}")